# Lecția 08 - Modelul de Design Multi-Agent


## Configurare


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## De ce Sistemele Multi-Agent?

Sarcinile din lumea reală, cum ar fi planificarea unei călătorii, implică multe tipuri diferite de expertiză — logistică, cunoștințe locale, bugetare și altele. Un singur agent care încearcă să gestioneze totul devine rapid dificil de administrat.

Sistemele multi-agent rezolvă această problemă prin **specializare**: fiecare agent se concentrează pe o zonă de expertiză, producând rezultate de calitate superioară față de un generalist. De asemenea, ele îmbunătățesc **scalabilitatea** — poți adăuga noi agenți (de exemplu, un specialist în zboruri, un critic de restaurante) fără a rescrie fluxul de lucru existent. Agenții se compun împreună printr-un flux structurat, trecând contextul de la unul la altul.


## Crearea agenților specializați


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Construirea unui flux de lucru secvențial

`WorkflowBuilder` vă permite să conectați agenți într-un graf orientat. Aici creăm o conductă simplă în două etape: **TravelPlanner** întocmește itinerariul, apoi **TravelConcierge** îl revizuiește și îl îmbunătățește.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Adăugarea Mai Multor Agenți în Fluxul de Lucru

Unul dintre cele mai mari avantaje ale modelului multi-agent este cât de ușor este de extins. Mai jos adăugăm un agent **BudgetReviewer** care verifică planul în raport cu bugetul călătorului, marchează elementele care ar putea depăși limita de cost și sugerează alternative pentru a economisi bani. Fluxul de lucru rulează acum trei agenți în secvență:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

În această lecție ai învățat cum să:

1. **Creezi agenți specializați** — fiecare cu un rol bine definit (planificare, concierge, revizuire buget).
2. **Conectezi agenții într-un flux de lucru secvențial** folosind `WorkflowBuilder` și `add_edge`.
3. **Transmiți în flux output-ul** dintr-un pipeline cu mai mulți agenți, urmărind care agent vorbește.
4. **Extinzi un flux de lucru** adăugând noi agenți în lanț fără a modifica pe cei existenți.

Modelul multi-agent menține fiecare agent simplu, în timp ce produce rezultate mai bogate și mai bine revizuite decât ar putea realiza un singur agent de unul singur.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare de responsabilitate**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un specialist uman. Nu ne asumăm răspunderea pentru eventualele neînțelegeri sau interpretări greșite rezultate din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
